#### Environment Check

In [1]:
import sys
print(sys.executable)

/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

load_dotenv("../.env")

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

#### Load Lesson Files

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

len(documents)

72

#### Q1

In [4]:
q1_answer = len(documents)
q1_answer

72

#### Build minsearch index

In [5]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

#### Search for Q2

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query,
    num_results=5,
)

q2_answer = results[0]["filename"]
q2_answer

'01-agentic-rag/lessons/14-agentic-loop.md'

#### Build RAG for full lesson pages

In [7]:
INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the course lesson pages.
Use only the facts from the CONTEXT.
If the answer is not in the CONTEXT, say you don't know.
""".strip()

PROMPT_TEMPLATE = """
QUESTION:
{question}

CONTEXT:
{context}
""".strip()


class LessonRAG:
    def __init__(
        self,
        index,
        llm_client,
        model="gpt-5.4-mini",
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
    ):
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.prompt_template = prompt_template

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")

            if "start" in doc:
                lines.append(f"Start offset: {doc['start']}")

            lines.append(doc["content"])
            lines.append("")

        return "\n\n".join(lines).strip()

    def build_prompt(self, question, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=question,
            context=context,
        )

    def llm(self, user_prompt):
        messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": user_prompt},
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=messages,
        )

        return response

    def rag(self, question):
        search_results = self.search(question)
        prompt = self.build_prompt(question, search_results)
        response = self.llm(prompt)

        return response.output_text, response.usage

#### Q3

In [8]:
assistant = LessonRAG(
    index=index,
    llm_client=openai_client,
    model=MODEL,
)

answer, usage = assistant.rag(query)

print(answer)

q3_answer = usage.input_tokens
q3_answer

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It calls the model with the current `messages`.
- If the model returns a function call, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls` is still `False`, the loop breaks.

So the stop condition is: **no function calls this turn means the agent is done**.


7122

#### Chunk lesson pages

In [9]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

len(chunks)

295

#### Q4

In [10]:
q4_answer = len(chunks)
q4_answer

295

#### Build minsearch index for chunks

In [11]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

chunk_index.fit(chunks)

#### Q5

In [12]:
chunk_assistant = LessonRAG(
    index=chunk_index,
    llm_client=openai_client,
    model=MODEL,
)

chunk_answer, chunk_usage = chunk_assistant.rag(query)

full_input_tokens = usage.input_tokens
chunk_input_tokens = chunk_usage.input_tokens

ratio = full_input_tokens / chunk_input_tokens

if ratio < 1.5:
    q5_answer = "about the same"
elif ratio < 6:
    q5_answer = "3x fewer"
elif ratio < 20:
    q5_answer = "10x fewer"
else:
    q5_answer = "30x fewer"

print(chunk_answer)
print("Full-page input tokens:", full_input_tokens)
print("Chunked input tokens:", chunk_input_tokens)
print("Ratio:", ratio)

q5_answer

It keeps calling the model inside a `while True` loop and checks whether any `function_call` items were returned.

- If the model returns a function call, the code runs the tool, appends the result to `messages`, and keeps looping.
- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls on that turn**.
Full-page input tokens: 7122
Chunked input tokens: 2340
Ratio: 3.0435897435897434


'3x fewer'

#### Build search tool for agent

In [13]:
import json

search_call_count = 0


def search(query: str) -> list[dict]:
    """
    Search the course lesson chunks for passages matching the given query.
    """
    global search_call_count
    search_call_count = search_call_count + 1

    return chunk_index.search(
        query,
        num_results=5,
    )

#### Define search tool schema

In [14]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lesson chunks for passages matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lesson chunks.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

#### Function call handler

In [15]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown tool: {call.name}")

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, indent=2),
    }

#### Agentic loop

In [16]:
def agent_loop(question, instructions, model="gpt-5.4-mini", max_iterations=10):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    answer = None

    for i in range(max_iterations):
        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        has_function_call = False

        for item in response.output:
            if item.type == "function_call":
                messages.append(make_call(item))
                has_function_call = True

            if item.type == "message":
                answer = item.content[0].text

        if not has_function_call:
            break

    return answer, messages

#### Q6

In [17]:
agent_instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

search_call_count = 0

agent_answer, agent_messages = agent_loop(
    question=agent_question,
    instructions=agent_instructions,
    model=MODEL,
)

q6_answer = search_call_count

print(agent_answer)
print("Search calls:", q6_answer)

q6_answer

The **agentic loop** is a repeat-until-done pattern:

1. Send the user prompt to the LLM.
2. The LLM may return a **tool call** instead of a final answer.
3. Your code runs that tool.
4. You send the tool result back to the LLM.
5. Repeat until the model produces a final answer with **no more tool calls**.

In the course, this was described as a `while` loop that “called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer” — that’s the foundation of agent frameworks.

### How that differs from plain RAG

**Plain RAG** is usually a fixed pipeline:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

So plain RAG is basically:

- retrieve once
- stuff the retrieved context into the prompt
- generate one answer

### Main difference

- **RAG**: retrieval happens as a single, preplanned step before generation.
- **Agenti

4